In [1]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_ta as ta
import math
from tqdm import tqdm
import gc
import time
import json

import optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)

# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline

from position_tools import calculate_trades, calculate_positions

In [11]:
dump_fname = "strat_results/DONCH--q_USDT--tf_1h--rsmpl_2_4_8--ntra_300--trra_0.4--mxlag_150--mxlkb_150--fst_100.txt"
with open(f'{dump_fname}', 'r') as file:
    # Read all lines into a list
    btso = [json.loads(l) for l in file.readlines()[:-1]]
# btso = [ {'asset':o['asset', 'timeframe': o['timeframe']]} for o in btso]

# df = pd.DataFrame([{'asset':o['asset'], 'timeframe': o['timeframe'], **{f'{k}_TRAIN':v for k,v in o['res_train'].items()},  **{f'{k}_TEST':v for k,v in o['res_test'].items()},  **o['params'] } for o in btso]).sort_values('tot_return_TEST')
# df.to_csv(f'donch-split-backtests-{train_ratio}.csv')
df = pd.DataFrame([{'asset':o['asset'], 'timeframe': o['timeframe'], **{f'{k}':v for k,v in o['res_test'].items()},  **o['params'] } for o in btso])
df.loc[:,['tot_return', 'asset_return', 'avg_win', 'avg_loss',
       'trade_max_drawdown']] = df.loc[:,['tot_return', 'asset_return', 'avg_win', 'avg_loss',
       'trade_max_drawdown']].apply(np.expm1)
df.loc[:,['tot_return', 'asset_return', 'avg_win', 'avg_loss',
       'trade_max_drawdown', 'win_ratio', 'profit_factor','band_offset']] = df.loc[:,['tot_return', 'asset_return', 'avg_win', 'avg_loss',
       'trade_max_drawdown', 'win_ratio', 'profit_factor','band_offset']].round(2)
df.sort_values('tot_return',ascending=False).to_csv(f'analyze-backtsts.csv')

In [7]:
df.columns

Index(['asset', 'timeframe', 'tot_return', 'asset_return', 'n_trades', 'nwins',
       'nlosses', 'win_ratio', 'profit_factor', 'avg_win', 'avg_loss',
       'trade_max_drawdown', 'up_lookback', 'dn_lookback', 'up_lag', 'dn_lag',
       'band_offset'],
      dtype='object')

In [39]:
df[df['tot_return_pct_TEST']>5].round(2) #.describe()

,asset,timeframe,tot_return_TRAIN,tot_return_pct_TRAIN,asset_return_pct_TRAIN,asset_return_TRAIN,n_trades_TRAIN,nwins_TRAIN,nlosses_TRAIN,win_ratio_TRAIN,...,sortino_TEST,avg_win_TEST,avg_loss_TEST,trade_max_drawdown_TEST,trade_max_drawdown_pct_TEST,up_lookback,dn_lookback,up_lag,dn_lag,band_offset
46,BTC,6,1.87,5.47,1.32,0.84,6,3,4,0.50,...,0.01,0.31,-0.09,-0.39,-0.32,183,146,66,111,0.30
2933,SAND,3,-0.49,-0.39,12.14,2.58,20,5,16,0.25,...,0.01,0.27,-0.10,-0.66,-0.48,122,108,57,66,0.38
2823,SHIB,1,0.78,1.18,-0.30,-0.36,14,6,9,0.43,...,0.01,0.20,-0.03,-0.25,-0.22,365,191,63,325,0.14
3555,ZRX,3,1.69,4.43,0.54,0.43,19,11,9,0.58,...,0.01,0.29,-0.10,-0.84,-0.57,92,125,99,129,0.35
1053,YFI,1,0.37,0.45,4.37,1.68,27,13,15,0.48,...,0.01,0.24,-0.06,-0.46,-0.37,199,166,111,150,0.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3436,TEL,24,1.38,2.97,-0.97,-3.59,8,3,6,0.38,...,0.14,1.00,-0.12,-1.37,-0.75,15,9,15,8,0.07
3435,TEL,24,1.55,3.72,-0.97,-3.59,7,2,6,0.29,...,0.16,0.97,-0.11,-0.99,-0.63,16,15,15,9,0.02
3433,TEL,24,1.38,2.97,-0.97,-3.59,8,3,6,0.38,...,0.13,1.04,-0.12,-1.45,-0.77,15,9,15,9,0.07
3429,TEL,12,1.40,3.07,-0.97,-3.59,11,3,9,0.27,...,0.08,0.82,-0.09,-0.74,-0.52,32,19,14,24,0.24


In [49]:
df = pd.json_normalize([{'bactest': {'asset':o['asset'], 'timeframe': o['timeframe']}, 'TRAIN': {f'{k}':v for k,v in o['res_train'].items()},  'TEST':{f'{k}':v for k,v in o['res_test'].items()},  'params': o['params'] } for o in btso]).round(3)

# Convert the column headers to a MultiIndex
df.columns = pd.MultiIndex.from_tuples([tuple(col.split('.')) for col in df.columns])
df.to_csv(f'donch-split-backtests-{train_ratio}.csv')
df


bactest                TRAIN                                  \
       asset timeframe tot_return tot_return_pct asset_return_pct   
0        ETH         1      3.153         22.408           -0.315   
1        ETH         1      2.716         14.126           -0.315   
2        ETH         1      3.136         22.010           -0.315   
3        ETH         1      2.336          9.337           -0.315   
4        ETH         1      2.906         17.283           -0.315   
...      ...       ...        ...            ...              ...   
4964    CELR        32      1.508          3.519           -0.840   
4965     FOR         1      0.564          0.758           -0.833   
4966     FOR         1      0.082          0.085           -0.833   
4967     FOR         1      0.024          0.024           -0.833   
4968     FOR         1     -0.294         -0.255           -0.833   

                                                    ...    TEST          \
     asset_return n_trades nwins nlosses win_ratio  ... sortino avg_win   
0          -0.378       45    23      23     0.511  ...   0.004   0.150   
1          -0.378       46    20      27     0.435  ...   0.006   0.185   
2          -0.378       73    29      45     0.397  ...   0.004   0.099   
3          -0.378       44    18      27     0.409  ...   0.006   0.186   
4          -0.378       53    22      32     0.415  ...   0.005   0.137   
...           ...      ...   ...     ...       ...  ...     ...     ...   
4964       -1.832       23     8      16     0.348  ...   0.152   1.049   
4965       -1.792       57    16      42     0.281  ...   0.007   0.068   
4966       -1.792       22     7      16     0.318  ...   0.003   0.096   
4967       -1.792       65    19      47     0.292  ...   0.005   0.055   
4968       -1.792       96    29      68     0.302  ...   0.012   0.065   

                                                             params  \
     avg_loss trade_max_drawdown trade_max_drawdown_pct up_lookback   
0      -0.052             -0.898                 -0.593         179   
1      -0.050             -0.574                 -0.437         174   
2      -0.041             -1.210                 -0.702         135   
3      -0.050             -0.721                 -0.514         179   
4      -0.048             -0.669                 -0.488         179   
...       ...                ...                    ...         ...   
4964   -0.120             -1.129                 -0.677           8   
4965   -0.037             -1.915                 -0.853          83   
4966   -0.065             -1.756                 -0.827          46   
4967   -0.029             -1.424                 -0.759         159   
4968   -0.025             -1.845                 -0.842          21   

                                            
     dn_lookback up_lag dn_lag band_offset  
0            199    102    200       0.265  
1            194    153    146       0.338  
2            174     48    132       0.199  
3            193    145    146       0.573  
4            194     78    156       0.304  
...          ...    ...    ...         ...  
4964          12     12     11       0.858  
4965         180     28    123       0.461  
4966         391    221    140       0.484  
4967         246     28    240       0.480  
4968         246     28     58       0.500  

[4969 rows x 37 columns]